[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C34_Agent_Orchestration_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身（MockLLM + orchestrator-worker）

本课全程 **纯标准库、CPU 可跑、无需任何 API key**。凡是需要「模型」的地方都用 **MockLLM**——一个确定性的假模型——再用 `assert` 验证编排与运维逻辑。

这个 notebook 做五件事：① 确认环境；② 认识 **orchestrator-worker** 结构；③ 造出本课的主角 **MockLLM**；④ 用它跑一个最小的「派生 subagent → 收结果」；⑤ 立下全课纪律——**不变量 + assert**，并展示**真实 Claude 无 key 回退**的适配形状。

## 1 · 环境自检

只需要标准库。`numpy` / `matplotlib` 可选（本课不强制）。**全程不联网、不需要 API key。**

In [ ]:
import sys, platform, json, time, re, heapq, dataclasses
print('Python', sys.version.split()[0], '|', platform.system())
try:
    import numpy as np; print('numpy', np.__version__, '(可选)')
except Exception:
    print('numpy 未安装（可选，不影响课程）')
print('标准库就绪：json/time/re/heapq/dataclasses 都在')
print('无需 API key —— 本课用 MockLLM。环境就绪 ✅')

## 2 · orchestrator-worker：多 agent 的基本结构

单个 agent 是一个 ReAct 回路。一个**多 agent 系统**的主流结构是 **orchestrator-worker**：
一个 orchestrator **分解**任务、**派活**给多个 worker（subagent）、**收集**结果、**合成**最终答案。

先把这个结构的骨架写出来——用最简单的「worker」和「分解 / 合成」占位。

In [ ]:
def orchestrate(task, split_fn, worker_fn, synth_fn):
    '''最小 orchestrator-worker 骨架。
       split_fn(task) -> [子任务...]；worker_fn(子任务) -> 结果；synth_fn(task, [结果...]) -> 最终答案。'''
    subtasks = split_fn(task)              # ① 分解
    results = [worker_fn(st) for st in subtasks]   # ② 派活给 worker（这里顺序模拟，真实可并行）
    return synth_fn(task, results)         # ③ 合成

# 一个玩具任务：把一句话按逗号拆成几段，每段交给一个 worker 数字数，最后汇总总字数
def split_by_comma(task):
    return [seg.strip() for seg in task.split('，') if seg.strip()]
def count_chars(seg):
    return {'seg': seg, 'n': len(seg)}
def sum_chars(task, results):
    return {'total': sum(r['n'] for r in results), 'parts': len(results)}

out = orchestrate('今天天气晴，适合出游，记得带水', split_by_comma, count_chars, sum_chars)
print('分解 + 派活 + 合成 ->', out)
assert out['parts'] == 3, '应分解成 3 个子任务'
assert out['total'] == 4 + 4 + 5, '总字数应是各段之和'
print('✅ orchestrator-worker 骨架跑通：分解 → 派活给 worker → 合成')

## 3 · 造出本课的主角：MockLLM

真实多 agent 里，每个 agent（orchestrator 与 subagent）的大脑都是大模型。本课用 **MockLLM** 代替它：一个**确定性、规则驱动**的假模型，对给定输入返回**结构正确**的输出。能力的不确定性被剥离，剩下的全是我们编排逻辑的对错——最适合学编排与运维。

下面这个 MockLLM 用「关键词→响应」规则表驱动，并记录被调次数（便于断言成本 / 调用数）。

In [ ]:
class MockLLM:
    '''确定性假模型：按规则把 prompt 映射到响应文本。
       规则 = [(关键词, 响应), ...]，第一个命中的生效；都不命中走 default。
       记录 calls 次数与累计「token」(用字符数粗略模拟)，供成本断言。'''
    def __init__(self, rules, default='(no rule matched)'):
        self.rules = rules
        self.default = default
        self.calls = 0
        self.total_tokens = 0
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        self.total_tokens += len(text)              # 粗略：输入字符数当 token
        for kw, resp in self.rules:
            if kw in text:
                self.total_tokens += len(resp)
                return resp
        self.total_tokens += len(self.default)
        return self.default

llm = MockLLM(rules=[
    ('总结', '这是一段摘要。'),
    ('翻译', 'This is a translation.'),
])
print('总结任务 ->', llm('请总结这段话'))
print('翻译任务 ->', llm('请翻译这段话'))
print('未命中   ->', llm('随便说点啥'))
print('调用次数 =', llm.calls, '| 累计 token =', llm.total_tokens)
assert llm('请总结一下') == '这是一段摘要。'
assert llm.calls == 4 and llm.total_tokens > 0
print('✅ MockLLM 工作正常：确定性、可断言、记录调用数与 token（供成本聚合）')

## 4 · 最小 subagent：派生一个有隔离上下文的 worker

subagent 的精髓是**隔离的上下文**：每个 subagent 有自己独立的对话历史，互不污染。
下面派两个 subagent 处理两个子任务，验证它们的上下文**真的隔离**（一个的历史不出现在另一个里），结果**结构化回传**。

In [ ]:
def run_subagent(subagent_id, task, llm):
    '''一个最小 subagent：拥有自己独立的 messages（隔离上下文），跑完回传结构化结果。'''
    messages = []                                  # ← 每次调用都是全新、隔离的上下文
    messages.append({'role': 'user', 'content': task})
    answer = llm(task)                             # 真实里: llm(messages)
    messages.append({'role': 'assistant', 'content': answer})
    return {'subagent_id': subagent_id, 'task': task,
            'result': answer, 'n_messages': len(messages)}

llm2 = MockLLM(rules=[('天气', '晴，26 度。'), ('股价', '收于 100 元。')])
r_a = run_subagent('A', '查一下天气', llm2)
r_b = run_subagent('B', '查一下股价', llm2)
print('subagent A ->', r_a)
print('subagent B ->', r_b)
# 上下文隔离：A 的任务/结果不应出现在 B 的结果里
assert r_a['result'] == '晴，26 度。' and r_b['result'] == '收于 100 元。'
assert '天气' not in r_b['task'] and '股价' not in r_a['task']
assert r_a['n_messages'] == 2 and r_b['n_messages'] == 2   # 各自独立的 2 条历史
print('✅ subagent：隔离上下文 + 结构化结果回传（模块 01 把它做扎实）')

## 5 · 立纪律：不变量 + assert（本课的对拍）；真实 Claude 无 key 回退

本课的裁判是**不变量**：编排 / 权限 / 追踪 / 队列在合法输入上做对、在非法输入上**正确报错或拒绝**。

先把这套工作流跑通：写一个极简的「fan-out 结果数不变量」检查，再展示**真实 Claude 无 key 自动回退**的适配形状（本系列承诺：无 key 绝不阻断）。

In [ ]:
def fan_out(subtasks, worker_fn):
    '''扇出：对每个子任务派一个 worker（顺序模拟并发），收集结果。
       铁律：返回的结果数必须等于子任务数（部分失败也一个不少）。'''
    results = []
    for st in subtasks:
        try:
            results.append({'task': st, 'ok': True, 'result': worker_fn(st)})
        except Exception as e:
            results.append({'task': st, 'ok': False, 'error': str(e)})  # 失败也回传一个结果
    return results

def flaky_worker(st):
    if st == 'boom':
        raise ValueError('worker 失败')
    return st.upper()

res = fan_out(['a', 'boom', 'c'], flaky_worker)
for r in res:
    print(r)
assert len(res) == 3, '结果数必须等于子任务数（fan-out 不变量）'
assert res[1]['ok'] is False and res[0]['ok'] is True
print('✅ fan-out 不变量：结果数==子任务数，单个失败被隔离成一个失败结果')

In [ ]:
# —— 真实 Claude 无 key 自动回退的适配形状（本环境无 key，走 MockLLM 分支）——
import os
def make_llm():
    if os.environ.get('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic                  # 仅在有 key 时导入
        client = Anthropic()
        def call(prompt):
            r = client.messages.create(model='claude-opus-4-8', max_tokens=512,
                                       messages=[{'role': 'user', 'content': prompt}])
            return ''.join(b.text for b in r.content if b.type == 'text')
        return call, 'real-claude(claude-opus-4-8)'
    # 无 key -> 确定性 MockLLM，课程照常跑通、绝不阻断
    return MockLLM(rules=[('你好', '你好，我是回退的 MockLLM。')]), 'MockLLM(fallback)'

llm_real_or_mock, backend = make_llm()
print('当前后端:', backend)
print('调用结果:', llm_real_or_mock('你好'))
assert backend.startswith('MockLLM') or backend.startswith('real')
print('✅ 无 key 自动回退到 MockLLM；有 key 则用真实 claude-opus-4-8 —— 形状一致、可迁移')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个零件（subagent 派生 / 隔离 / 扇出、orchestrator 编排、权限沙箱、span 树与成本聚合、任务队列与重试）都用**不变量 + assert** 验证；逻辑正确则 assert 通过，assert 通过则可把 `MockLLM(...)` 换成真实 `messages.create(model='claude-opus-4-8', ...)` 直接迁移。

**接下来五个模块**：01 subagents → 02 orchestration → 03 permissions → 04 observability → 05 deploy。每一步都建立在「orchestrator-worker」这张图上。

下一站：**模块 01 · 子 Agent（Subagents）**。